# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available record sets, fields, and columns. Only entities with @id are referenced.
record_sets = []
for rs in metadata.record_sets:
    print(f"RecordSet name: {rs.name}, @id: {rs.id}")
    record_sets.append(rs.id)
    if hasattr(rs, 'fields'):
        print('  Fields:')
        for field in rs.fields:
            print(f"    Field name: {getattr(field, 'name', 'n/a')}, @id: {getattr(field, 'id', 'n/a')}")
            if hasattr(field, 'column'):
                print(f"      Column @id: {getattr(field.column, 'id', 'n/a')}")
    print('-' * 60)
if not record_sets:
    print("No record sets found in the Croissant schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set using its @id
dfs = {}
for record_set_id in record_sets:
    print(f"Reading records from record set {record_set_id} ...")
    recs = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(recs)
    dfs[record_set_id] = df
    print(f"  - columns: {df.columns.tolist()}")
    print(df.head(2))

# For further analysis, pick the first available record set (if any found)
if record_sets:
    main_record_set_id = record_sets[0]
    print(f"Selected main record set for EDA: {main_record_set_id}")
    print(dfs[main_record_set_id].columns)

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# List available numeric columns in the main DataFrame (if any)
main_df = dfs[main_record_set_id]
numeric_cols = main_df.select_dtypes(include=[np.number]).columns.tolist()
print(f"Numeric columns: {numeric_cols}")

if numeric_cols:
    numeric_field = numeric_cols[0]
    print(f"Using numeric field: {numeric_field}")
    # Threshold to filter (use median as threshold if >0)
    threshold = main_df[numeric_field].median() if main_df[numeric_field].median() > 0 else 1
    filtered_df = main_df[main_df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold} (showing 5 records):")
    print(filtered_df.head())

    # Normalized field
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"\nNormalized {numeric_field} for filtered records (showing 5 records):")
    print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
    
    # Try grouping by first non-numeric column
    group_fields = [c for c in main_df.columns if c not in numeric_cols]
    if group_fields:
        group_field = group_fields[0]
        print(f"\nAttempting groupby: {group_field}")
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index().sort_values(by=numeric_field, ascending=False)
        print(grouped_df.head())
else:
    print("No numeric columns found to process.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# Plot histogram of main numeric field if available
if numeric_cols:
    main_df[numeric_field].hist(bins=50)
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.title(f'Histogram of {numeric_field}')
    plt.show()
    
    # Boxplot by group_field if exists
    if 'group_field' in locals():
        filtered_df.boxplot(column=numeric_field, by=group_field, grid=False, rot=90)
        plt.title(f'{numeric_field} grouped by {group_field}')
        plt.suptitle('')
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No numeric columns to plot.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.